# H18a companion: toy LR robustness and step-1 spectral-bound diagnostic

This notebook is the analysis/reporting counterpart of:

- `experiments/Tier_1_Core_Mechanism_Tests/H18a_MUON_LR_STABILITY_SPECTRAL_BOUND/run_experiment.py`

It **does not reimplement the experiment logic**. Instead, it imports the script's `run_experiment()` function, runs the same toy study, and analyzes the returned structured results.

## Truthful scope

This is a **toy deep linear-network study**. It is meant to examine whether Muon appears more learning-rate-robust than momentum SGD across depth, and whether a **normalized step-1 operator-norm diagnostic** is consistent with that robustness.

### Calibration

- The reported LRs are **best grid LRs** from a finite discrete sweep, **not** exact stability boundaries.
- The mechanistically relevant quantity in this notebook is **`||ΔW||_op / lr`**, not raw `||ΔW||_op` by itself.
- The notebook supports a **toy-scope, evidence-consistent interpretation**, not a proof or universal claim.


## 1. Imports, notebook-safe path handling, and script loading

The original notebook had a `__file__` usage that is not valid in a normal notebook kernel. This pass removes that blocker by locating the experiment directory explicitly from `Path.cwd()` and its parents.


In [ ]:
from pathlib import Path
import importlib.util
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

EXPERIMENT_REL = Path('experiments/Tier_1_Core_Mechanism_Tests/H18a_MUON_LR_STABILITY_SPECTRAL_BOUND')


def locate_experiment_paths():
    cwd = Path.cwd().resolve()

    for base in [cwd, *cwd.parents]:
        candidate = base / EXPERIMENT_REL / 'run_experiment.py'
        if candidate.exists():
            return {
                'repo_root': base,
                'experiment_dir': candidate.parent,
                'script_path': candidate,
            }

    for base in [cwd, *cwd.parents]:
        candidate = base / 'run_experiment.py'
        if candidate.exists() and candidate.parent.name == 'H18a_MUON_LR_STABILITY_SPECTRAL_BOUND':
            return {
                'repo_root': candidate.parent.parents[2],
                'experiment_dir': candidate.parent,
                'script_path': candidate,
            }

    raise FileNotFoundError(
        'Could not locate H18a_MUON_LR_STABILITY_SPECTRAL_BOUND/run_experiment.py from the current working directory.'
    )


PATHS = locate_experiment_paths()
REPO_ROOT = PATHS['repo_root']
EXPERIMENT_DIR = PATHS['experiment_dir']
SCRIPT_PATH = PATHS['script_path']

spec = importlib.util.spec_from_file_location('h18a_muon_lr_stability_spectral_bound', SCRIPT_PATH)
h18a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h18a)

print(f'Repository root: {REPO_ROOT}')
print(f'Experiment dir : {EXPERIMENT_DIR}')
print(f'Script path    : {SCRIPT_PATH}')
print('Loaded run_experiment() from script successfully.')


## 2. Reproducibility and runtime configuration

By default, this notebook runs the **full script-default experiment**. For automated smoke checks only, an environment variable can request a much smaller run:

- set `H18A_NOTEBOOK_SMOKE=1` to execute a reduced configuration quickly
- leave it unset for the real toy-study results

This keeps the default scientific setup intact while making notebook-path verification practical.


In [ ]:
SMOKE_MODE = os.environ.get('H18A_NOTEBOOK_SMOKE', '0') == '1'

if SMOKE_MODE:
    run_kwargs = dict(
        depths=[2, 3],
        sgd_lr_grid=h18a.SGD_LR_GRID[:4],
        muon_lr_grid=h18a.MUON_LR_GRID[:4],
        seeds=[42, 179],
        selection_seed_count=1,
        num_steps=20,
        verbose=False,
    )
else:
    run_kwargs = dict(verbose=False)

print(f'Smoke mode: {SMOKE_MODE}')
print(f'run_kwargs: {run_kwargs}')

start_time = time.perf_counter()
payload = h18a.run_experiment(**run_kwargs)
elapsed_s = time.perf_counter() - start_time

config = payload['config']
results_table = payload['results']['table']
analysis = payload['analysis']

print(f'Completed run in {elapsed_s:.2f} s')
print(f'Returned {len(results_table)} depth/optimizer records')


## 3. Configuration log

This section records the exact experiment configuration returned by the script, along with notebook execution metadata.


In [ ]:
config_rows = [
    ('script_path', str(SCRIPT_PATH)),
    ('repo_root', str(REPO_ROOT)),
    ('smoke_mode', SMOKE_MODE),
    ('elapsed_s', round(elapsed_s, 3)),
    ('dim', config['dim']),
    ('depths', config['depths']),
    ('num_steps', config['num_steps']),
    ('momentum', config['momentum']),
    ('ns_iters', config['ns_iters']),
    ('batch_size', config['batch_size']),
    ('seeds', config['seeds']),
    ('selection_seed_count', config['selection_seed_count']),
    ('estimated_training_runs', config['estimated_training_runs']),
    ('sgd_lr_grid_range', (config['sgd_lr_grid'][0], config['sgd_lr_grid'][-1], len(config['sgd_lr_grid']))),
    ('muon_lr_grid_range', (config['muon_lr_grid'][0], config['muon_lr_grid'][-1], len(config['muon_lr_grid']))),
    ('scope_note', config['scope_note']),
    ('mechanism_note', config['mechanism_note']),
]

config_df = pd.DataFrame(config_rows, columns=['field', 'value'])
display(config_df)


**Interpretation.**

- In normal mode, this notebook runs the same toy protocol as the script.
- In smoke mode, the shapes of the returned objects and figures remain the same, but the numerical results should **not** be interpreted scientifically.
- The script remains the single source of truth for experiment logic; the notebook is now a reporting layer over structured outputs.


## 4. Structured result tables

We first convert the script's returned dictionaries into compact tables for analysis and plotting.


In [ ]:
summary_rows = []
sweep_rows = []

for record in results_table:
    summary_rows.append({
        'depth': record['depth'],
        'optimizer': record['optimizer'],
        'best_lr': record['best_lr'],
        'best_selection_mean_final_loss': record['best_selection_mean_final_loss'],
        'mean_final_loss': record['mean_final_loss'],
        'std_final_loss': record['std_final_loss'],
        'mean_max_step1_op_norm': record['mean_max_step1_op_norm'],
        'std_max_step1_op_norm': record['std_max_step1_op_norm'],
        'mean_avg_step1_op_norm': record['mean_avg_step1_op_norm'],
        'std_avg_step1_op_norm': record['std_avg_step1_op_norm'],
        'mean_max_step1_op_norm_over_lr': record['mean_max_step1_op_norm_over_lr'],
        'mean_avg_step1_op_norm_over_lr': record['mean_avg_step1_op_norm_over_lr'],
        'finite_final_loss_count': record['finite_final_loss_count'],
    })

    for sweep_item in record['sweep']:
        sweep_rows.append({
            'depth': record['depth'],
            'optimizer': record['optimizer'],
            'lr': sweep_item['lr'],
            'selection_mean_final_loss': sweep_item['selection_mean_final_loss'],
            'selection_finite_count': sweep_item['selection_finite_count'],
        })

summary_df = pd.DataFrame(summary_rows).sort_values(['depth', 'optimizer']).reset_index(drop=True)
sweep_df = pd.DataFrame(sweep_rows).sort_values(['optimizer', 'depth', 'lr']).reset_index(drop=True)

fit_df = pd.DataFrame([
    {
        'optimizer': opt,
        'slope': analysis['fits'][opt]['slope'],
        'r2': analysis['fits'][opt]['r2'],
        'lr_ratio': analysis['fits'][opt]['lr_ratio'],
        'min_best_lr': analysis['fits'][opt]['min_best_lr'],
        'max_best_lr': analysis['fits'][opt]['max_best_lr'],
    }
    for opt in ['sgd', 'muon']
])

verdict_rows = []
for key, test in analysis['verdict']['tests'].items():
    verdict_rows.append({
        'test': key,
        'description': test['description'],
        'value': test['value'],
        'comparison': test['comparison'],
        'threshold': test['threshold'],
        'pass': test['pass'],
    })
verdict_df = pd.DataFrame(verdict_rows)

summary_display = summary_df.copy()
for col in [
    'best_lr', 'best_selection_mean_final_loss', 'mean_final_loss', 'std_final_loss',
    'mean_max_step1_op_norm', 'std_max_step1_op_norm',
    'mean_avg_step1_op_norm', 'std_avg_step1_op_norm',
    'mean_max_step1_op_norm_over_lr', 'mean_avg_step1_op_norm_over_lr',
]:
    summary_display[col] = summary_display[col].map(lambda x: float(f'{x:.4e}'))

fit_display = fit_df.copy()
for col in ['slope', 'r2', 'lr_ratio', 'min_best_lr', 'max_best_lr']:
    fit_display[col] = fit_display[col].map(lambda x: float(f'{x:.4e}'))

verdict_display = verdict_df.copy()
for col in ['value', 'threshold']:
    verdict_display[col] = verdict_display[col].map(lambda x: float(f'{x:.4e}'))

display(summary_display)
display(fit_display)
display(verdict_display)


**Interpretation.**

The summary table is the main compact record of what the script actually computes:

- a discrete LR sweep for each `(depth, optimizer)` pair
- the selected **best grid LR**
- final-loss statistics at that selected LR
- step-1 operator-norm statistics
- normalized step-1 operator norms, which are the most meaningful mechanism-facing quantities in this toy study

The fit table should be read cautiously: the slopes are descriptive fits to a coarse discrete grid, not formal exponents.


## 5. LR sweep outcomes by depth and optimizer

Each curve shows the selection-phase mean final loss as a function of LR for one depth. This is the actual surface from which the notebook/script selects the best grid LR.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sorted(summary_df['depth'].unique()))))

for ax, optimizer in zip(axes, ['sgd', 'muon']):
    opt_sweep = sweep_df[sweep_df['optimizer'] == optimizer]
    opt_summary = summary_df[summary_df['optimizer'] == optimizer]

    for color, depth in zip(colors, sorted(opt_sweep['depth'].unique())):
        depth_df = opt_sweep[opt_sweep['depth'] == depth].copy()
        finite_df = depth_df[np.isfinite(depth_df['selection_mean_final_loss']) & (depth_df['selection_mean_final_loss'] > 0)]
        if finite_df.empty:
            continue

        ax.plot(
            finite_df['lr'],
            finite_df['selection_mean_final_loss'],
            marker='o',
            linewidth=1.8,
            markersize=4,
            color=color,
            label=f'L={depth}',
        )

        best_row = opt_summary[opt_summary['depth'] == depth].iloc[0]
        ax.scatter(
            [best_row['best_lr']],
            [best_row['best_selection_mean_final_loss']],
            color=color,
            edgecolor='black',
            s=90,
            zorder=5,
        )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Learning rate')
    ax.set_ylabel('Selection-phase mean final loss')
    ax.set_title(f'{optimizer.upper()} LR sweep')
    ax.legend(loc='best', fontsize=9)

fig.suptitle('Discrete LR sweep outcomes used to choose the best grid LR', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


**Interpretation.**

This plot is important because it grounds the later claims in the actual selection procedure. The notebook is not estimating a continuous stability boundary; it is selecting the lowest-loss point from a finite candidate grid. Any narrative about “stability” or “optimal LR” should therefore be read as **best-grid behavior under this protocol**.


## 6. Best-grid LR versus depth

We now compare how the selected best grid LR changes with depth for SGD and Muon.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_specs = {
    'sgd': {'color': '#d62728', 'marker': 'o'},
    'muon': {'color': '#1f77b4', 'marker': 's'},
}

for optimizer, spec in plot_specs.items():
    opt_df = summary_df[summary_df['optimizer'] == optimizer].sort_values('depth')
    fit = analysis['fits'][optimizer]
    ax.plot(
        opt_df['depth'],
        opt_df['best_lr'],
        marker=spec['marker'],
        color=spec['color'],
        linewidth=2.2,
        label=f"{optimizer.upper()} best grid LR (slope={fit['slope']:.3f}, ratio={fit['lr_ratio']:.1f}x)",
    )
    ax.plot(
        fit['depths'],
        fit['predicted_best_lrs'],
        linestyle='--',
        color=spec['color'],
        alpha=0.8,
        label=f'{optimizer.upper()} log-log fit',
    )

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Depth L')
ax.set_ylabel('Best grid learning rate')
ax.set_title('Depth dependence of the selected best grid LR')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()


**Interpretation.**

The key qualitative comparison is the contrast in shrinkage:

- SGD's selected best grid LR falls strongly with depth.
- Muon's selected best grid LR also decreases, but much more modestly.

This notebook therefore supports a **relative LR-robustness** claim. It does **not** support calling Muon “flat in depth” if the observed fitted slope is clearly away from zero.


## 7. Normalized step-1 max operator norm versus depth

The most informative mechanism-facing metric in this toy study is:

`mean max ||ΔW||_op / best_lr`

If Muon's update orthogonalization keeps the operator norm of the pre-LR update near 1, this normalized quantity should stay near 1, while SGD can grow with depth.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for optimizer, spec in plot_specs.items():
    opt_df = summary_df[summary_df['optimizer'] == optimizer].sort_values('depth')
    ax.plot(
        opt_df['depth'],
        opt_df['mean_max_step1_op_norm_over_lr'],
        marker=spec['marker'],
        color=spec['color'],
        linewidth=2.2,
        label=optimizer.upper(),
    )

ax.axhline(1.0, color='black', linestyle='--', linewidth=1.4, label='reference: 1.0')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Depth L')
ax.set_ylabel(r'Mean max step-1 $||\Delta W||_{op} / lr$')
ax.set_title('Normalized step-1 update operator norm across depth')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


**Interpretation.**

This is the clearest correction relative to the earlier overclaim. The meaningful toy diagnostic is **not** that raw `||ΔW||_op` stays constant at the selected LR. Raw `||ΔW||_op` naturally changes when the selected LR changes.

Instead, the more serious question is whether `||ΔW||_op / lr` stays controlled. In this toy setup, Muon's normalized value staying near 1 is the strongest evidence in favor of a step-1 spectral-bound explanation.


## 8. Compact T1/T2/T3 practical verdict table

These are the current practical robustness criteria already used by the script. They remain useful as compact summary tests, but they should be interpreted as pragmatic thresholds rather than theorem statements.


In [ ]:
display(verdict_display)


In [ ]:
sgd_fit = analysis['fits']['sgd']
muon_fit = analysis['fits']['muon']

pass_string = ', '.join(
    [f"{name.split('_')[0]}={'PASS' if test['pass'] else 'FAIL'}" for name, test in analysis['verdict']['tests'].items()]
)

conclusion_md = f"""
## 9. Calibrated conclusion

In this toy deep linear-network study:

- **SGD** best-grid LR shrank by **{sgd_fit['lr_ratio']:.1f}x** across the tested depths, with exploratory log-log slope **{sgd_fit['slope']:.3f}**.
- **Muon** best-grid LR shrank by **{muon_fit['lr_ratio']:.1f}x**, with slope **{muon_fit['slope']:.3f}**.
- Muon is therefore **substantially less depth-sensitive than SGD** in this protocol, but the observed Muon slope is **not near zero**, so it should not be described as depth-flat.
- The strongest mechanism-consistent evidence in this pair is the **normalized step-1 max operator norm** `||ΔW||_op / lr`: Muon stays near the `O(1)` reference level while SGD grows strongly with depth.
- The T1/T2/T3 thresholds evaluate to **{pass_string}**.

### What this notebook supports

This notebook supports the claim that, **in this toy setting**, Muon is more learning-rate-robust than momentum SGD and that a normalized step-1 spectral-bound diagnostic is consistent with that robustness.

### What this notebook does not support

It does **not** establish:

- a true continuous stability boundary,
- a universal or exact `O(1/L^2)` law for SGD,
- depth-invariant Muon LR,
- or a formal proof that the spectral-bound mechanism is the unique causal explanation.
"""

display(Markdown(conclusion_md))


## 10. Recommended next verification steps

For a stronger second pass, the most useful upgrades would be:

1. add confidence intervals or seed-level error bars to LR and normalized operator-norm summaries,
2. retain and visualize per-layer step-1 operator norms directly,
3. replace coarse best-grid LR language with a more explicit boundary-search procedure if a true stability claim is desired,
4. connect the spectral-bound story to additional mechanism diagnostics beyond step 1.
